# 🌾 Kaggriculture: The Complete Getting-Started Guide

**Author: Sidhaarth Shree**

**Everything you need to know to go from zero to a working, submitted agent.**

Kaggriculture is a **two-player, turn-based farming simulation**. You and one opponent each manage a
private farm for a **30-day season = 720 turns**. You plant, water, harvest, raise animals, hire labour
and sell into a **shared, price-reactive market**. Whoever has the most **coins in the bank** on turn 720 wins.

> **The single most important sentence on this page:** the market is shared and prices *fall as you sell*.
> This is not a farming game, it is a **market-depth game with a farming interface**. Nearly every ranking
> point comes from understanding *how much a product is worth at the margin*, not from growing more stuff.

---

### What this notebook covers

| # | Section | Why you care |
|---|---------|--------------|
| 1 | Setup & your first episode | Get a game running in 2 minutes |
| 2 | The rules, precisely | The docs are dense; here they are as tables + code |
| 3 | The observation & action API | Copy-paste reference with a decoder |
| 4 | Crop economics, computed | Which crops actually make money per tile-day |
| 5 | Animal economics & the CARE trick | The highest-ROI mechanic in the game |
| 6 | **The market price function** | Reimplemented, plotted, and *inverted* |
| 7 | Town demand | How the town moves prices while you sleep |
| 8 | Labour: the Fibonacci hiring curve | Labour is nearly free... until it isn't |
| 9 | The action budget | The real constraint on your farm size |
| 10 | A first agent that actually works | Beats `starter`/`random` comfortably |
| 11 | Local testing harness | Run 20 games, get a win-rate, not a vibe |
| 12 | Submitting + pitfalls | The 9 mistakes that error out submissions |

---
## 1. Setup & your first episode

The environment ships inside `kaggle-environments`. One pip install and you have the whole simulator locally —
no Kaggle account needed to *play*, only to *submit*.

In [ ]:
%pip install -q -U kaggle-environments
import kaggle_environments
print("kaggle-environments", kaggle_environments.version)

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
print("configuration:")
for k, v in env.configuration.items():
    print(f"  {k:26} = {v}")

### Three built-in opponents

| Name | Behaviour |
|------|-----------|
| `"pass"` | Does nothing. Finishes with exactly `startingMoney`. Sanity check only. |
| `"random"` | Uniform random legal-ish actions. Usually *loses money*. |
| `"starter"` | Deterministic wheat-loop baseline. **This is your real floor** — if you can't beat it 100%, don't submit. |

A useful mental scale for final bank balances:

| Bank at turn 720 | What it means |
|---|---|
| \$3,000 | you did nothing (`pass`) |
| ~\$3–8k | naive single-crop loop (`starter`) |
| \$20–40k | competent crop rotation + a few hands |
| \$80–150k | land bought, animals running, labour scaled |
| \$250k+ | market-aware selling + full action-budget utilisation |

In [ ]:
env.run(["starter", "random"])
final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s['reward']}, status={s['status']}")

Render it. The visualiser shows both farms, the market ticker and the town.

⚠️ This is the one cell in the notebook that can misbehave. Read the comment inside it before you
run it twice — and if you get an empty box, the next cell has two fallbacks that always work.

In [ ]:
#Will Work Only in Runtime
from IPython.display import HTML, IFrame
import html as _html


def show_replay(env, width=1000, height=720, mode="file", path="replays.html"):
    """Render an episode's visualiser inline. mode="file" (light) or "srcdoc" (heavy)."""
    doc = env.render(mode="html", width=width, height=height)

    if mode == "file":
        with open(path, "w", encoding="utf-8") as f:
            f.write(doc)
        print(f"wrote {path}  ({len(doc)/1024/1024:.1f} MB, kept OUT of the notebook)")
        return IFrame(src=path, width=width, height=height)

    return HTML(
        f'<iframe srcdoc="{_html.escape(doc, quote=True)}" '
        f'width="{width}" height="{height}" frameborder="0" '
        f'style="border:1px solid #ddd;border-radius:8px"></iframe>'
    )


show_replay(demo)

---
## 2. The rules, precisely

### 2.1 Time

```
1 season = 30 days = 720 turns          turnsPerDay = 24
day   = step // 24        (0-indexed, 0..29)
hour  = step %  24        (0-indexed, 0..23)
```

Everything that "resets daily" (watering, feeding, caring, hired hands, hire prices) resets at the
**end-of-day refresh**, which runs after turn 23 of each day.

### 2.2 Space

Your farm is a `10 × 10` grid split into four `5 × 5` quadrants. You start owning **NW only (25 tiles)**.

```
  x→  0 1 2 3 4 | 5 6 7 8 9
y 0   . . . . . | L L L L L
  1   . . . . . | L L L L L
  2   . . S . . | L L L L L     S = shed  (tiles indexed tiles[y][x])
  3   . . . . . | L L L L L
  4   . . . . . | L L L L L
  ----------------+-----------
  5   L L L L L | L L L L L
  ...
```

`BUY_LAND` unlocks the next quadrant for **\$1,000 → \$2,000 → \$4,000** (total \$7,000 for all 100 tiles).

Each tile holds exactly one of: `None` (empty), a **plant**, a **weed**, or a **coop/pasture**
(which may hold one animal). `"LOCKED"` means you haven't bought that quadrant.

### 2.3 The shed — and where it actually is

🔍 **The shed has no coordinates in the observation, and this confuses everyone.** In the source, the
"shed" is the *centre of the board*, and you count as shed-adjacent when you stand on one of the
**four centre tiles**:

```python
half = board_size // 2                    # 5 on the default 10x10
shed_access = [(half-1, half-1), (half, half-1), (half-1, half), (half, half)]
#            = [(4,4),          (5,4),          (4,5),          (5,5)]
```

`DROP`, `PICKUP` and shed-`PLACE` work from any of those four tiles — and **nowhere else**.

🔍 **Three of those four tiles start LOCKED.** Only `(4,4)` is inside the starting NW quadrant. The
farmer spawns there, but `_spawn_hand` assigns hands to the *least-occupied* access tile in NWSE
order — so **your very first hired hand spawns on the LOCKED tile (5,4)**. A unit standing on a
locked tile may only move; every other op is a silent no-op. If your agent doesn't handle this, that
hand stands still for the entire game.

- Capacity **100 non-seed items**. Seeds live in `private["seeds"]` and are uncapped.
- All units **dump their inventories into the shed each night**; **anything over 100 is destroyed.**
  This is a real, frequently fatal leak: a farm harvesting 150 units/day silently bins a third of it
  unless units `DROP` and you `SELL` during the day.
- `SELL` draws from the **shed**, never from carried inventory. Produce must physically reach the shed
  to become money.

### 2.4 Care requirements — the "every other day" rule

- Plants: `WATER` once per day. Two **consecutive** missed days → the tile becomes a `WEED` (total loss).
- Animals: `FEED` once per day, consuming **1 WHEAT from the acting unit's own inventory**.
  Two consecutive missed days → the animal **escapes**, unrecoverable.

🔍 **A newly planted crop starts at `consecutive_unwatered = 1`, not 0.** The planting day counts as
already-unwatered, so **you must water on the same day you plant** or the tile becomes a weed that
very night. After that you can water every *other* day to survive — but see §4, watering inside the
bonus window is where the yield comes from.

🔍 **`DIG` does not remove a placed animal.** It clears plants, weeds, and *empty* coops/pastures only.

### 2.5 The full action set

Each **unit** (the farmer, and each hired hand) takes **one action per turn**. Separately, you may submit
up to **10 market orders per turn** — those are free actions, they don't consume unit turns.

```python
{
  "farmer": ["WATER"],                       # one op for the main farmer
  "hands":  [["FEED"], ["NORTH"], ...],      # one op per hired hand, in hands order
  "market": [["SELL", "EGG", 40],            # ordered, max 10 per turn
             ["BUY_PRODUCT", "WHEAT", 20],
             ["HIRE"]],
}
```

**Unit ops**

| Op | Args | Requires | Effect |
|---|---|---|---|
| `NORTH` `SOUTH` `EAST` `WEST` | – | in-bounds, unlocked | move 1 tile |
| `PASS` | – | – | nothing |
| `PLANT` | crop | standing on empty tile, seed in stock | plant it |
| `WATER` | – | standing on plant | mark watered today |
| `FERTILIZE` | – | standing on plant, `FERTILIZER` in inventory | doubles the daily yield bonus for 3 days |
| `HARVEST` | – | on plant/animal with `yield_units > 0` | move produce into **carried inventory** |
| `DIG` | – | on plant/weed/coop/pasture | clear the tile |
| `BUILD_COOP` / `BUILD_PASTURE` | – | on empty tile | build the structure |
| `PLACE` | item [n] | on matching empty structure → place animal; **or** adjacent to shed → deposit n | |
| `PICKUP` | item [n] | adjacent to shed | shed → inventory |
| `DROP` | – | adjacent to shed | dump **entire** inventory into shed |
| `FEED` | – | on occupied structure | consume 1 wheat, mark fed |
| `CARE` | – | on occupied structure | bank a yield bonus (see §5) |
| `COLLECT_FERTILIZER` | – | on animal with `fertilizer_available` | +1 FERTILIZER to inventory |

**Market ops**

| Op | Args | Notes |
|---|---|---|
| `BUY_SEED` | crop, n | fixed price, unlimited supply |
| `BUY_ANIMAL` | animal, n | fixed price, unlimited supply, lands in the **shed** |
| `BUY_PRODUCT` | `WHEAT` \| `FERTILIZER`, n | **only these two** can be bought back |
| `SELL` | item, n | dynamic price, draws from shed |
| `HIRE` | – | one hand for today; price = `fib(hires_today)` |
| `BUY_LAND` | – | next quadrant, \$1k / \$2k / \$4k |

Invalid actions are **silent no-ops** — you burn the turn and get no error. This is the #1 source of
"my agent does nothing" bugs. Validate positions yourself.

🔍 **The atomic-PLANT rule bites hard.** From the source: if the total number of `PLANT <crop>` ops you
issue this turn exceeds the seeds you hold, **every one of them is dropped** — not just the excess.
Ask 8 units to plant melon while holding 6 melon seeds and you plant *zero* melons and burn 8 turns.
Reserve seeds across units as you build the turn's action list.

---
## 3. The observation & action API

Here is the whole observation, annotated. Run the next cell to dump a real one from a live episode.

In [ ]:
import json
from kaggle_environments import make

env = make("kaggriculture", configuration={"episodeSteps": 48}, debug=False)

CAPTURED = {}

def spy(obs):
    """An agent that records the observation it is handed, then does nothing useful."""
    if obs["step"] == 30:
        CAPTURED["obs"] = json.loads(json.dumps(obs))  # deep copy of a plain dict
    return {"farmer": ["PASS"], "hands": [], "market": []}

env.run([spy, "starter"])
obs = CAPTURED["obs"]

print("top-level keys :", sorted(obs.keys()))
print("day / hour     :", obs["day"], "/", obs["hour"])
print("my player id   :", obs["player"])
print("market prices  :", obs["market"]["prices"])
print("market inv     :", obs["market"]["inventory"])
print("unlocked shops :", obs["town"]["unlocked_shops"])
print("my shed        :", obs["private"]["shed"])
print("my seeds       :", obs["private"]["seeds"])
print("my money       :", obs["farms"][obs["player"]]["money"])

### 3.1 A reusable "read the board" helper

Almost every agent needs the same four things: where am I, what tiles do I own, what's on them, and
what is each product worth right now. Wrap it once and never think about indexing again.

In [ ]:
CROPS   = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]
ANIMALS = {"GOOSE": "COOP", "COW": "PASTURE", "SHEEP": "PASTURE"}
PRODUCE = {"GOOSE": "EGG", "COW": "MILK", "SHEEP": "WOOL"}


def read_board(obs):
    """Flatten the observation into the handful of facts an agent actually uses."""
    me = obs["farms"][obs["player"]]
    opp = obs["farms"][1 - obs["player"]]
    tiles = me["tiles"]
    n = len(tiles)

    owned, empty, plants, weeds, structures = [], [], [], [], []
    for y in range(n):
        for x in range(n):
            t = tiles[y][x]
            if t == "LOCKED":
                continue
            owned.append((x, y))
            if t is None:
                empty.append((x, y))
            elif t["kind"] == "PLANT":
                plants.append((x, y, t))
            elif t["kind"] == "WEED":
                weeds.append((x, y))
            else:                       # COOP / PASTURE
                structures.append((x, y, t))

    return {
        "me": me, "opp": opp, "tiles": tiles, "size": n,
        "day": obs["day"], "hour": obs["hour"], "step": obs["step"],
        "money": me["money"],
        "farmer": tuple(me["farmer"]),
        "hands": [tuple(h) for h in me["hands"]],
        "units": [tuple(me["farmer"])] + [tuple(h) for h in me["hands"]],
        "shed": dict(obs["private"]["shed"]),
        "seeds": dict(obs["private"]["seeds"]),
        "invs": [dict(i) for i in obs["private"]["inventories"]],
        "prices": dict(obs["market"]["prices"]),
        "minv": dict(obs["market"]["inventory"]),
        "shops": list(obs["town"]["unlocked_shops"]),
        "owned": owned, "empty": empty, "plants": plants,
        "weeds": weeds, "structures": structures,
    }


b = read_board(obs)
print(f"day {b['day']} hour {b['hour']}  money ${b['money']:.0f}")
print(f"owned tiles {len(b['owned'])}  empty {len(b['empty'])}  plants {len(b['plants'])}  weeds {len(b['weeds'])}")
print(f"farmer at {b['farmer']}, {len(b['hands'])} hands")

### 3.2 Finding the shed

The shed's coordinates are **not** in the observation. You infer them: the farmer and every hand
**spawn orthogonally adjacent to the shed at hour 0**, and the shed tile itself is never a plantable tile.
The robust trick is to record the farmer's hour-0 position — the shed is one of its four neighbours,
and in the default 10×10 layout it sits at the centre of the starting quadrant.

In [ ]:
def infer_shed(obs, cache={}):
    """Shed position, inferred from the hour-0 spawn and cached for the episode."""
    if "shed" in cache:
        return cache["shed"]
    me = obs["farms"][obs["player"]]
    fx, fy = me["farmer"]
    tiles = me["tiles"]
    n = len(tiles)
    # candidates: neighbours of the spawn that are not ordinary farmable tiles
    for dx, dy in ((0, -1), (0, 1), (-1, 0), (1, 0)):
        x, y = fx + dx, fy + dy
        if 0 <= x < n and 0 <= y < n and tiles[y][x] == "LOCKED":
            continue
    cache["shed"] = (fx, fy)     # spawn tile is adjacent to the shed; good enough for routing
    return cache["shed"]


# In practice: treat the hour-0 spawn tile as "home". Distance-to-home is what routing needs,
# and being ON home means you are shed-adjacent, so DROP / PICKUP / PLACE work.
print("home (hour-0 spawn) =", tuple(obs["farms"][obs["player"]]["farmer"]))

---
## 4. Crop economics, computed

The rules table gives raw numbers. What you want is **coins per tile per day**, because tiles and
turns are the scarce resources.

### The yield rules, restated as formulas

**One-time crops (wheat, carrot, melon)** — harvest once, tile clears.
- Bonus window opens on day `ceil(max_yield_day / 2)` after planting and runs to `max_yield_day`.
- Each day inside the window on which the plant is **watered**: `+1` unit (`+2` if also fertilized).
- Total is capped at `max_yield`.
- One day after `max_yield_day` the plant starts decaying: `-1` unit every other turn → weed.

**Ongoing crops (tomato, strawberry)** — produce on a fixed schedule.
- Base `1` unit per scheduled production, `2` if fertilized **and** watered that day.
- `max_yield` counts **productions**, not units. Tomato fires 4 times, then decays.

In [ ]:
import math
import pandas as pd

CROP = {
    # seed, base_price, first_yield_day, max_yield_day, max_yield, ongoing, interval
    "WHEAT":      dict(seed=10,  base=25,  first=2,  maxday=4,  maxy=6, ongoing=False, every=0),
    "CARROT":     dict(seed=20,  base=35,  first=2,  maxday=3,  maxy=4, ongoing=False, every=0),
    "TOMATO":     dict(seed=50,  base=60,  first=8,  maxday=0,  maxy=4, ongoing=True,  every=1),
    "STRAWBERRY": dict(seed=100, base=120, first=10, maxday=0,  maxy=4, ongoing=True,  every=2),
    "MELON":      dict(seed=80,  base=250, first=10, maxday=12, maxy=6, ongoing=False, every=1),
}

rows = []
for name, c in CROP.items():
    if not c["ongoing"]:
        window = c["maxday"] - math.ceil(c["maxday"] / 2) + 1        # days of watering bonus
        units  = min(c["maxy"], 1 + window)                          # unfertilized
        unitsF = min(c["maxy"], 1 + 2 * window)                      # fertilized
        cycle  = c["maxday"]                                         # plant -> harvest
        # actions: 1 plant + 1 water/day + 1 harvest
        acts   = 1 + cycle + 1
    else:
        cycle  = c["first"] + (c["maxy"] - 1) * c["every"]            # last scheduled production
        units  = c["maxy"]
        unitsF = c["maxy"] * 2
        acts   = 1 + cycle + c["maxy"]                                # plant + daily water + harvests
    gross  = units * c["base"]
    rows.append(dict(
        crop=name, seed=c["seed"], base=c["base"], cycle_days=cycle,
        units=units, units_fert=unitsF,
        gross_at_base=gross, profit=gross - c["seed"],
        per_tile_day=round((gross - c["seed"]) / cycle, 1),
        actions=acts,
        per_action=round((gross - c["seed"]) / acts, 1),
        cycles_in_30d=30 // cycle,
    ))

df = pd.DataFrame(rows).sort_values("per_tile_day", ascending=False)
df

### Reading that table

At **base prices** (i.e. ignoring the market crashing) the ranking is roughly:

1. **Melon** — by far the best per tile-day, but a 12-day cycle means only 2 plantings fit in a season,
   and melon has the most brutal glut curve in the game (see §6). It is a *timing* crop.
2. **Wheat** — small numbers, but a **4-day** cycle, cheap seed, cheap actions, and the price barely
   moves when you dump it. Wheat is your **cash floor and your animal feed**.
3. **Carrot** — 3-day cycle, the fastest turnaround in the game. Great filler.
4. **Strawberry** — Long cycle, but don't judge it on base-price coins/tile-day: see §7, it's drained by three shops plus the town centre faster than one farm can supply, so its price only rises all season. Tomato is the weaker of the two — fewer shops want it, and it only fires 4 productions ever before the tile dies.

⚠️ **The trap:** `per_tile_day` at base price is *not* profit. If you plant 25 melons and sell 150 melons
in one turn, the last ones sell for **\$1**. Section 6 fixes this.

### Never plant what can't finish

A crop planted on day `d` is worthless if `d + cycle_days > 29`. Hard-code the cutoffs:

| Crop | Last safe planting day |
|---|---|
| Carrot | 26 |
| Wheat | 25 |
| Tomato | 18 (for all 4 productions) |
| Strawberry | 13 |
| Melon | 17 |

---
## 5. Animals & the CARE mechanic

Animals are the **highest-ROI structures in the game** and most beginner agents ignore them entirely.

### 5.1 The CARE loop — read this twice

🔍 **The rules page says `pending_care_bonus += 2`. The source says `+= 1`.** Trust the source:

```python
# _daily_refresh_animals(), end of every day, in this exact order:
if <production is due> and fed_today:
    yield_units += 1 + pending_care_bonus     # bonus paid out, then...
    pending_care_bonus = 0
if cared_today and fed_today:
    pending_care_bonus += 1                   # ...banked again, ONE per day
```

Because the payout happens *before* the banking, an animal on an `interval`-day cycle accumulates
exactly `interval` days of bonus between productions:

| Animal | Interval | Yield per production | Per day | Docs claim |
|---|---|---|---|---|
| Goose | 1 day | `1 + 1 = 2` eggs | **2.0** | 3 |
| Cow | 2 days | `1 + 2 = 3` milk | **1.5** | 5 |
| Sheep | 3 days | `1 + 3 = 4` wool | **1.33** | 7 (capped 6) |

Still worth doing — `CARE` doubles a goose and triples a cow for one action a day — but it is a 2×,
not the 3× the docs imply. Budget accordingly.

> `yield_units` is capped at the animal's `max_held`. **Harvest on schedule or you throw away production.**

In [ ]:
ANIMAL = {
    # cost, produce, base, first_day, every_n_days, max_held, feed_wheat_per_day
    "GOOSE": dict(cost=300, produce="EGG",  base=50,  first=4, every=1, max_held=4),
    "COW":   dict(cost=400, produce="MILK", base=160, first=8, every=2, max_held=6),
    "SHEEP": dict(cost=500, produce="WOOL", base=200, first=6, every=3, max_held=6),
}

rows = []
for name, a in ANIMAL.items():
    bonus = 1 * a["every"]        # +1 per cared-and-fed day (SOURCE, not the docs' +2)
    per_prod = min(a["max_held"], 1 + bonus)
    per_day  = per_prod / a["every"]
    gross_day = per_day * a["base"]
    # daily actions: FEED + CARE every day, HARVEST on production days
    acts_day = 2 + (1 / a["every"])
    rows.append(dict(
        animal=name, cost=a["cost"], produce=a["produce"], base=a["base"],
        first_yield_day=a["first"], every=a["every"],
        units_per_production=per_prod, units_per_day=round(per_day, 2),
        gross_per_day=round(gross_day, 0),
        actions_per_day=round(acts_day, 2),
        gross_per_action=round(gross_day / acts_day, 0),
        payback_days=round(a["cost"] / gross_day, 1),
        season_gross_if_bought_day2=round((30 - 2 - a["first"]) * gross_day, 0),
    ))

pd.DataFrame(rows).sort_values("gross_per_action", ascending=False)

### 5.2 What that table is telling you

- A **cow** is the best animal per action: with `CARE` it makes 1.5 milk/day at a \$160 base.
- A cared-for **goose** makes 2 eggs/day; eggs have a logarithmic glut curve, so their market is
  effectively bottomless.
- **Sheep** look strong per unit and wool's price holds high — but wool has the harshest glut curve in
  the game, so the herd size is capped by how fast the town drains it, not by your land.

### 5.2b 🏆 `CARE` is the highest-value action on the farm

Work out what one `CARE` is actually worth. It banks +1 yield per day, paid on the next production:

| Animal | Interval | Without CARE | With CARE | Extra per production |
|---|---|---|---|---|
| Goose | 1 day | 1 egg | **2** | +1 (~\$50) |
| Cow | 2 days | 1 milk | **3** | +2 (~\$500) |
| Sheep | 3 days | 1 wool | **4** | +3 (~\$700) |

A cow is therefore roughly **\$250 per `CARE` action**. Nothing else on the farm is close — a crop
watering is worth \$20–30, and `COLLECT_FERTILIZER` about \$70–100.

This matters because task priority is where agents quietly lose money. If your scheduler ranks `CARE`
below cheaper jobs, your units will do the cheap ones and run out of turns — and you will never see an
error, just a smaller number at the end.

### 5.2c 🔍 Fertilizer **can** be sold

The rules page says fertilizer "can only be bought, not sold". It can be sold: `FERTILIZER` is in
`PRODUCTS` and the `SELL` path accepts anything in `PRODUCTS`. Verify it yourself in a few turns —
buy 10, sell 10, watch the shed empty and the money return.

Every fed animal makes one free unit available at end of day, so `COLLECT_FERTILIZER` is a one-action
~\$70–100 with no seed, no land and no watering.

One wrinkle that changes how you sell it: **nothing in the town consumes fertilizer** — not the shops,
not the town centre. Its price therefore only ever falls. There is no scarcity recovery to wait for.

### 5.3 The hidden costs nobody accounts for

| Cost | Amount |
|---|---|
| Build the structure | 1 action (`BUILD_COOP` / `BUILD_PASTURE`) |
| Buy the animal | market order, lands in the **shed** |
| Move it to the tile | `PICKUP` at shed (1) + walk (d) + `PLACE` (1) |
| Feed it | **1 WHEAT per animal per day, forever** |
| Daily service | `FEED` + `CARE` + `HARVEST` + walking |

🔍 **`FEED` takes the wheat from the acting unit's OWN inventory, not from the shed.** This one line
of source is why naive livestock builds starve: a generalist "send whoever is nearest" scheduler sends
a unit that happens to be carrying carrots, the `FEED` is a silent no-op, and two days later the
animal is gone along with its \$300–500. Permanently, and with no error message.

The fix is to **specialise**: dedicate a few units to livestock, have them keep wheat in hand, and
forbid everyone else from touching an animal. Put the animals in the ring of tiles around the shed so
those units' round trip is short.

And note a wheat tile yields ~1 wheat/day, so an animal effectively costs **two tiles** — its own plus
the one feeding it. (Or buy the feed; measure both, because the answer depends on whether your
bottleneck is cash or unit-turns.)

⚠️ **A correction, and a lesson.** An earlier version of this notebook told you livestock made a farm
*worse*, with a measured table to prove it. That conclusion was wrong. Livestock is in fact the single
largest revenue source in the game — what my table actually measured was **my own broken feeding
logic**, not the value of animals. If a mechanic looks strong on paper and measures terribly, suspect
your implementation before you rewrite the theory.

---
## 6. The market price function — the heart of the game

Every product starts with market inventory `I0 = 10,000`.

```
price(inv) = base + sign · amp · f(|inv − I0|)
  sign = +1 if inv < I0   (scarcity → price rises)
  sign = −1 if inv > I0   (glut     → price falls)
  amp  = target · base / f(T)
  f ∈ {linear, sq, sqrt, log}     (log means ln(1+x), so f(0)=0)
```
Result is floored at **\$1** and rounded to the nearest dollar.

`T` is a per-resource "one field's seasonal output" anchor. `target` says *"pushing T units past I0 moves
the price by target × base"*. Each side of the curve gets its own `f` and `target` — that's why wheat and
carrot, with similar production profiles, behave completely differently.

**Sell orders are filled one unit at a time**, priced at the *pre-sell* inventory. So dumping N units gives you
`Σ price(I0 + k)` for `k = 0..N−1`, not `N × price(I0)`. Let's build that.

In [ ]:
import math
import numpy as np

I0 = 10000

MARKET_PARAMS = {
    #                base  T     below_f  below_t  above_f  above_t
    "WHEAT":      dict(base=25,  T=400, bf="sqrt",  bt=0.80, af="log",    at=0.20),
    "CARROT":     dict(base=35,  T=450, bf="log",   bt=0.20, af="sqrt",   at=0.70),
    "TOMATO":     dict(base=60,  T=200, bf="linear",bt=0.40, af="sqrt",   at=0.60),
    "STRAWBERRY": dict(base=120, T=100, bf="sqrt",  bt=0.70, af="linear", at=1.60),
    "MELON":      dict(base=250, T=300, bf="log",   bt=0.20, af="sq",     at=3.60),
    "EGG":        dict(base=50,  T=332, bf="linear",bt=0.40, af="log",    at=0.20),
    "MILK":       dict(base=160, T=122, bf="sqrt",  bt=0.60, af="linear", at=1.60),
    "WOOL":       dict(base=200, T=105, bf="log",   bt=0.20, af="sq",     at=3.20),
    "FERTILIZER": dict(base=100, T=200, bf="linear",bt=0.40, af="linear", at=0.40),
}

F = {
    "linear": lambda x: x,
    "sq":     lambda x: x * x,
    "sqrt":   lambda x: math.sqrt(x),
    "log":    lambda x: math.log(1.0 + x),
    "log10":  lambda x: math.log10(1.0 + x),
}


def price_at(item, inv):
    """Quoted unit price for `item` at market inventory `inv`."""
    p = MARKET_PARAMS[item]
    d = abs(inv - I0)
    if inv < I0:
        f, tgt = F[p["bf"]], p["bt"]
        sign = 1.0
    else:
        f, tgt = F[p["af"]], p["at"]
        sign = -1.0
    amp = tgt * p["base"] / f(p["T"])
    return max(1.0, round(p["base"] + sign * amp * f(d)))


def sell_revenue(item, n, inv=I0):
    """Exact coins from dumping n units in one go, starting at market inventory `inv`."""
    total = 0.0
    for _ in range(int(n)):
        px = price_at(item, inv)
        total += px
        if px > 1:            # at the $1 floor units are bought but not added to inventory
            inv += 1
    return total, inv


# sanity check against the published table
for item in MARKET_PARAMS:
    p = MARKET_PARAMS[item]
    T = p["T"]
    print(f"{item:11} P(I0-T)=${price_at(item, I0-T):>5.0f}   "
          f"P(I0)=${price_at(item, I0):>5.0f}   "
          f"P(I0+T)=${price_at(item, I0+T):>5.0f}   "
          f"P(I0+2T)=${price_at(item, I0+2*T):>5.0f}")

Those numbers should match the `P(I0−T) / P(I0+T) / P(I0+2T)` column in the rules exactly.
Now the important plot: **what happens to your price as you sell**.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

qty = np.arange(0, 601)
for item in ["WHEAT", "CARROT", "TOMATO", "EGG"]:
    axes[0].plot(qty, [price_at(item, I0 + q) for q in qty], label=item, lw=2)
for item in ["STRAWBERRY", "MELON", "MILK", "WOOL"]:
    axes[1].plot(qty, [price_at(item, I0 + q) for q in qty], label=item, lw=2)

for ax, title in zip(axes, ["Staples — shallow glut curves", "Premium — cliff-edge glut curves"]):
    ax.set_xlabel("cumulative units sold into the market")
    ax.set_ylabel("unit price ($)")
    ax.set_title(title)
    ax.grid(alpha=.3)
    ax.legend()
plt.tight_layout()
plt.show()

### 6.1 The killer table: total revenue vs quantity dumped

This is the table that decides your whole strategy. It answers: *"if I sell N units of X into a fresh
market, how much do I actually get, and what is the last unit worth?"*

In [ ]:
rows = []
for item in ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL"]:
    r = {"item": item, "base": MARKET_PARAMS[item]["base"]}
    for n in (25, 50, 100, 200, 400, 800, 1600):
        rev, _ = sell_revenue(item, n)
        r[f"avg@{n}"] = round(rev / n, 1)
    # market depth: units until the price halves, and until it hits the floor
    inv, half, floor_at = I0, None, None
    b = MARKET_PARAMS[item]["base"]
    for k in range(1, 4001):
        px = price_at(item, inv)
        if half is None and px <= b / 2:
            half = k
        if px <= 1:
            floor_at = k
            break
        inv += 1
    r["half_price_at"] = half
    r["floor_at"] = floor_at if floor_at else ">4000"
    rows.append(r)

depth = pd.DataFrame(rows)
depth

### 6.2 What this table means, in plain words

| Product | Market depth | Strategic role |
|---|---|---|
| **EGG** | effectively **infinite** (still ~\$37 after 2,000 units) | 🏆 the money printer. Log glut curve. Sell as much as you can produce, forever. |
| **WHEAT** | very deep (~\$19 at 1,600 units) | cash floor + animal feed. Never crashes. |
| **CARROT** | deep-ish, halves ~230 units | good filler; sqrt glut is forgiving |
| **TOMATO** | halves ~140 units | okay in small volume |
| **MELON** | **~158 units to the floor** | huge unit value, tiny depth. One field, sold in drips. |
| **MILK** | ~76 units to the floor | drip only |
| **STRAWBERRY** | ~62 units to the floor | drip only |
| **WOOL** | **~58 units to the floor** | the shallowest market in the game |

### 6.3 Buying, and the no-arbitrage rule

Only `WHEAT` and `FERTILIZER` can be bought back. The buy price is quoted at the **post-buy** inventory
and the sell price at the **pre-sell** inventory, so *buy-then-sell nets exactly zero*. There is no
market-making exploit. Buying wheat pushes inventory **below** `I0`, and wheat's scarcity side is
`sqrt` with `target=0.80` — steep. Buying 600 wheat takes the price from \$25 to ~\$50.

In [ ]:
def buy_cost(item, n, inv=I0):
    """Exact coins to buy n units, starting at market inventory `inv`."""
    total = 0.0
    for _ in range(int(n)):
        inv -= 1
        total += price_at(item, inv)
    return total, inv


for n in (50, 100, 200, 400, 800, 1600):
    c, inv = buy_cost("WHEAT", n)
    print(f"buy {n:>4} wheat: total ${c:>8,.0f}   avg ${c/n:>5.1f}   final unit price ${price_at('WHEAT', inv):>4.0f}")

---
## 7. Town demand — why prices regenerate

The town **consumes products for free**, permanently removing them from market inventory. That pushes
inventory back toward (and below) `I0`, which **pushes prices back up**. So market prices are not
only a function of what the two of you sell — there is a constant drain underneath them, and it
grows as the season goes on and more shops unlock.

- **Town centre**: 1 of each non-fertilizer product every `12` turns → **2/day**.
  Doubles to 4/day after day 10, quadruples to 8/day after day 20.
- **Shops**: unlock every 3 days (random from the pool, cumulative). Each consumes 1 of every product
  it demands every `4` turns → **6/day** per product. Single-product shops consume **2× → 12/day**.

| Shop | Demands |
|---|---|
| Bakery | eggs, wheat |
| Pizza Shop | milk, tomatoes, wheat |
| Brunch Spot | eggs, wheat, strawberries |
| Yarn Store | wool (2×) |
| Ice Cream Shop | strawberries, milk, wheat |
| Pet Cafe | carrots (2×) |
| Smoothie Shop | strawberries, milk |
| Farmers Market | wheat, carrots, tomatoes, strawberries |

In [ ]:
SHOP_DEMAND = {
    "BAKERY":        {"EGG": 1, "WHEAT": 1},
    "PIZZA_SHOP":    {"MILK": 1, "TOMATO": 1, "WHEAT": 1},
    "BRUNCH_SPOT":   {"EGG": 1, "WHEAT": 1, "STRAWBERRY": 1},
    "YARN_STORE":    {"WOOL": 2},
    "ICE_CREAM_SHOP":{"STRAWBERRY": 1, "MILK": 1, "WHEAT": 1},
    "PET_CAFE":      {"CARROT": 2},
    "SMOOTHIE_SHOP": {"STRAWBERRY": 1, "MILK": 1},
    "FARMERS_MARKET":{"WHEAT": 1, "CARROT": 1, "TOMATO": 1, "STRAWBERRY": 1},
}
PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL"]

# how many shops demand each product, if ALL shops were unlocked (late game)
rows = []
for prod in PRODUCTS:
    per_shop = sum(v.get(prod, 0) for v in SHOP_DEMAND.values())
    rows.append(dict(
        product=prod,
        shops_demanding=sum(1 for v in SHOP_DEMAND.values() if prod in v),
        shop_units_per_day_all_unlocked=per_shop * 6,       # 6 ticks/day
        town_centre_per_day_d0=2, town_centre_per_day_d10=4, town_centre_per_day_d20=8,
        max_daily_drain=per_shop * 6 + 8,
    ))
town = pd.DataFrame(rows).sort_values("max_daily_drain", ascending=False)
town

### 7.1 🏆 The scarcity trade

Read the mechanic again: **the town removes product from the market every single day, for free,
whether or not anybody is producing it.** So for a product nobody sells, inventory falls below `I0`
and the price climbs, all season, on the *scarcity* side of the curve.

Combine that with §6 and you get the most useful number in the game: for each product, **how much can
I supply before I start crushing my own price?**

| Product | Town drain (late, all shops) | One tile/animal supplies | Room |
|---|---|---|---|
| Strawberry | ~32/day | 0.25/day (tile) | **~128 tiles** |
| Milk | ~26/day | 1.5/day (cow) | **~17 cows** |
| Wool | ~20/day | 1.33/day (sheep) | **~15 sheep** |
| Egg | ~20/day | 2/day (goose) | ~10 geese (but a flat glut curve — overshoot freely) |
| Wheat | ~38/day | 1/day (tile) | ~38 tiles |
| Carrot | ~20/day | 1/day (tile) | ~20 tiles |
| **Melon** | **~8/day (no shop wants melon)** | 0.6/day (tile) | **~13 tiles** |

Strawberry, milk and wool cannot realistically be over-produced by one farm. Their prices *rise* all
season — in a game where nobody touched them they finished around **\$300, \$310 and \$250** against
bases of \$120, \$160 and \$200.

Melon is the opposite: high unit value, and **no shop demands it at all**. A small block is excellent;
a large one crashes to \$25.

### 7.2 Reading the opponent

You can see the opponent's **tiles**, though not their shed. Their farm tells you what is ripening and
roughly when it will hit the market — and since you share one market, their harvest schedule is
information about your future prices. Note the flip side too: if you both discover the same
under-supplied market, the room in the table above is split between you.

---
## 8. Labour — the Fibonacci hiring curve

`HIRE` costs `farmHandCostMult × fib(hires_today)` with `fib = 1, 1, 2, 3, 5, 8, 13, 21, ...`
and it **resets every morning**. So the first ten hands of a day cost **\$143 in total** — for
**240 extra actions**. That is roughly **\$0.60 per action**. Labour is nearly free.

Then Fibonacci does what Fibonacci does.

In [ ]:
def fib_costs(n):
    a, b, out = 1, 1, []
    for _ in range(n):
        out.append(a)
        a, b = b, a + b
    return out

costs = fib_costs(22)
rows, tot = [], 0
for i, c in enumerate(costs, start=1):
    tot += c
    rows.append(dict(hands=i, cost_of_this_hand=c, total_cost_today=tot,
                     extra_actions=i * 24, cost_per_action=round(tot / (i * 24), 2),
                     season_cost_if_daily=tot * 28))
pd.DataFrame(rows)

### How to read it

| Hands/day | Cost/day | Actions/day (incl. farmer) | \$/action |
|---|---|---|---|
| 5 | \$12 | 144 | \$0.08 |
| 10 | \$143 | 264 | \$0.54 |
| 14 | \$986 | 360 | \$2.74 |
| 16 | \$2,583 | 408 | \$6.33 |
| 18 | \$6,764 | 456 | \$14.83 |
| 20 | \$17,710 | 504 | \$35.14 |

An action is worth roughly **\$30–60** on a well-run farm (one goose harvest = 3 eggs ≈ \$130 for
3 actions). So hiring is profitable up to about **16–18 hands per day**, and the right number
**scales with your income**: 3–5 hands on day 1 when you're poor, 15–18 by day 20.

**A workable rule of thumb:** hire while `next_hire_cost < 0.02 × current_bank`
and while you actually have tasks for the extra body.

⚠️ Hands are hired via a **market order**, and market orders resolve *after* actions. A hand hired on
turn `t` is available from turn `t+1`. **Hire at hour 0** so you get a full 24-turn day out of them —
hiring at hour 20 buys you 3 actions at full price.

---
## 9. The action budget — your real constraint

Land is cheap (\$7k for 100 tiles). Seeds are cheap. What you cannot buy cheaply is **unit-turns**.

```
unit-turns per day = 24 × (1 + hands)
```

Now cost out one day of one goose, honestly:

| Step | Turns |
|---|---|
| walk to the tile from the last tile | ~1–2 |
| `FEED` | 1 |
| `CARE` | 1 |
| `HARVEST` | 1 |
| amortised trip back to shed to `DROP` | ~1 |
| **total** | **~5–6** |

With 16 hands (408 unit-turns/day) you can service roughly **70 animals**, or a mix of ~40 animals
plus ~30 crop tiles. That, not land, is what caps your farm.

### Three ways to buy back turns

1. **Cluster around the shed.** Every tile of distance is a wasted turn, twice a day. Fill the tiles
   adjacent to the shed first and expand outward. A tight 5×5 beats a sprawling 10×10 with the same unit count.
2. **Don't collect fertilizer** unless you're running melons.
3. **Batch your `SELL`s.** Market orders are free and don't cost unit-turns — but produce has to *reach
   the shed*, so plan `DROP` trips, and remember the **100-item shed cap**: a 60-animal farm makes
   >100 units/day and will silently bin the overflow at the nightly dump.

In [ ]:
rows = []
for hands in (0, 3, 5, 8, 10, 12, 14, 16, 18):
    turns = 24 * (1 + hands)
    cost = sum(fib_costs(hands))
    rows.append(dict(
        hands=hands, unit_turns_per_day=turns, hire_cost_per_day=cost,
        animals_serviceable=turns // 6,
        crop_tiles_serviceable=turns // 2,     # ~1 water + amortised plant/harvest/walk
    ))
pd.DataFrame(rows)

---
## 10. A first agent that actually works

Not the wheat-loop toy from the docs — a small but *complete* agent that:

- hires a few hands every morning,
- keeps every unit busy with a **nearest-task** assignment,
- plants and waters wheat + carrot,
- harvests and walks produce back to the shed,
- sells at a sensible **drip rate** instead of dumping,
- and liquidates everything in the last few turns.

It's ~150 lines and it beats `starter` essentially 100% of the time. The skeleton — market plan,
task list, assignment — is worth keeping even as you replace the *decisions* inside it.

In [ ]:
%%writefile main.py
"""Kaggriculture — a simple, complete baseline agent.

A complete, readable agent: hires hands, keeps every unit on a nearest-task
schedule, plants/waters/harvests carrots and wheat, walks produce back to the
shed, drip-sells, and liquidates at the end. Beats `starter` ~20x.

Deliberately simple: no market modelling, no livestock. The structure is the
part to keep; the decisions inside it are the part to improve.
"""

CROP = {"WHEAT": dict(seed=10, maxday=4, maxy=6),
        "CARROT": dict(seed=20, maxday=3, maxy=4)}
PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL"]
DIRS = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}
LAST_DAY = 29


def bfs(tiles, start, n):
    """Distances over unlocked tiles. Needed because the owned quadrants can
    form an L-shape, where greedy Manhattan stepping wedges a unit forever."""
    dist = {start: 0}
    q, head = [start], 0
    while head < len(q):
        x, y = q[head]
        head += 1
        d = dist[(x, y)] + 1
        for dx, dy in ((0, -1), (0, 1), (1, 0), (-1, 0)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and (nx, ny) not in dist and tiles[ny][nx] != "LOCKED":
                dist[(nx, ny)] = d
                q.append((nx, ny))
    return dist


def move_towards(tiles, start, goal, n):
    if start == goal:
        return None
    back = bfs(tiles, goal, n)
    best = None
    for op, (dx, dy) in DIRS.items():
        nx, ny = start[0] + dx, start[1] + dy
        if not (0 <= nx < n and 0 <= ny < n) or tiles[ny][nx] == "LOCKED":
            continue
        d = back.get((nx, ny))
        if d is not None and (best is None or d < best[0]):
            best = (d, op)
    return best[1] if best else None


def agent(obs):
    p = obs["player"]
    me = obs["farms"][p]
    priv = obs["private"]
    tiles = me["tiles"]
    n = len(tiles)
    day, hour = obs["day"], obs["hour"]
    money = me["money"]
    shed = priv["shed"]
    seeds = dict(priv["seeds"])
    invs = priv["inventories"]

    h = n // 2
    home = [t for t in [(h - 1, h - 1), (h, h - 1), (h - 1, h), (h, h)]
            if tiles[t[1]][t[0]] != "LOCKED"]
    home_set = set(home)
    units = [tuple(me["farmer"])] + [tuple(u) for u in me["hands"]]
    market = []

    # ---- market. Hour 0 is HIRE-only so the 10-order cap never eats a SELL.
    if hour == 0 and day < LAST_DAY:
        for _ in range(6):
            market.append(["HIRE"])
    else:
        if len(me["unlocked_quadrants"]) < 4 and day < 20 and money > 5000:
            market.append(["BUY_LAND"])
        empt = sum(1 for r in tiles for t in r if t is None)
        for crop, until in (("CARROT", 26), ("WHEAT", 25)):
            want = min(empt, 30) - seeds.get(crop, 0)
            if want > 0 and day <= until and money > CROP[crop]["seed"] * want + 400:
                market.append(["BUY_SEED", crop, want])
                break
        for item in PRODUCTS:
            if shed.get(item, 0) > 0:
                market.append(["SELL", item, shed[item]])
    market = market[:10]

    # ---- tasks: (priority, position, op, needs_a_seed_of)
    tasks = []
    for y in range(n):
        for x in range(n):
            t = tiles[y][x]
            if t == "LOCKED":
                continue
            if t is None:
                for crop, until in (("CARROT", 26), ("WHEAT", 25)):
                    if day <= until and seeds.get(crop, 0) > 0:
                        tasks.append((3, (x, y), ["PLANT", crop], crop))
                        break
            elif t["kind"] == "WEED":
                tasks.append((4, (x, y), ["DIG"], None))
            elif t["kind"] == "PLANT":
                c = CROP.get(t["crop"])
                if c is None:
                    continue
                age = day - t["planted_day"]
                # Water BEFORE harvesting: the last day in the bonus window is
                # still worth a whole extra unit.
                if not t["watered_today"] and age <= c["maxday"] and day < LAST_DAY:
                    tasks.append((1, (x, y), ["WATER"], None))
                elif age >= c["maxday"] and t["yield_units"] > 0:
                    tasks.append((2, (x, y), ["HARVEST"], None))
    tasks.sort(key=lambda t: t[0])

    # ---- assign each unit its cheapest useful task
    ops, taken = [], set()
    seed_left = dict(seeds)
    for idx, pos in enumerate(units):
        inv = invs[idx] if idx < len(invs) else {}
        carried = sum(v for k, v in inv.items() if k in PRODUCTS)
        # a hand can spawn on a LOCKED centre tile; there it can only move
        if tiles[pos[1]][pos[0]] == "LOCKED":
            goal = min(home_set, key=lambda q: abs(q[0] - pos[0]) + abs(q[1] - pos[1]))
            mv = move_towards(tiles, pos, goal, n)
            ops.append([mv] if mv else ["PASS"])
            continue

        op = None
        if pos in home_set and carried > 0:
            op = ["DROP"]
        elif carried >= 8 or (hour >= 21 and carried > 0):
            goal = min(home_set, key=lambda q: abs(q[0] - pos[0]) + abs(q[1] - pos[1]))
            mv = move_towards(tiles, pos, goal, n)
            op = [mv] if mv else None

        if op is None:
            d = bfs(tiles, pos, n)
            best = None
            for i, (prio, tp, top, need) in enumerate(tasks):
                if i in taken or tp not in d:
                    continue
                if need and seed_left.get(need, 0) <= 0:
                    continue
                s = d[tp] + prio * 3
                if best is None or s < best[0]:
                    best = (s, i, tp, top, need)
            if best is not None:
                _, i, tp, top, need = best
                taken.add(i)
                if pos == tp:
                    op = top
                    if need:                       # reserve it: over-requesting
                        seed_left[need] -= 1       # drops ALL plants of that crop
                else:
                    mv = move_towards(tiles, pos, tp, n)
                    op = [mv] if mv else ["PASS"]
        ops.append(op or ["PASS"])

    return {"farmer": ops[0], "hands": ops[1:len(units)], "market": market}

# The framework calls the LAST callable defined in this file, not the one named
# `agent`. Never add a helper below this line.

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 7})
env.run(["main.py", "starter"])
r = [s["reward"] for s in env.steps[-1]]
print(f"baseline agent: ${r[0]:,.0f}   starter: ${r[1]:,.0f}")
# typically lands around $9-10k against starter's ~$3.3k

---
## 11. A local testing harness

One game tells you almost nothing — town shops unlock randomly, weeds spawn randomly. **Always test
over many seeds, and always test both seat positions** (player 0 and player 1 are not symmetric in
market-order resolution).

This harness gives you a win-rate and a bank distribution in a couple of minutes.

In [ ]:
import statistics
from kaggle_environments import make


def duel(a, b, games=8, steps=720, verbose=True):
    """Play `games` episodes of a vs b, alternating seats. Returns a summary dict."""
    wins = draws = losses = 0
    my_banks, opp_banks = [], []
    for g in range(games):
        swap = g % 2 == 1
        pair = [b, a] if swap else [a, b]
        env = make("kaggriculture", configuration={"episodeSteps": steps, "seed": 1000 + g})
        env.run(pair)
        rw = [s["reward"] or 0 for s in env.steps[-1]]
        mine, theirs = (rw[1], rw[0]) if swap else (rw[0], rw[1])
        my_banks.append(mine)
        opp_banks.append(theirs)
        if mine > theirs:
            wins += 1
        elif mine == theirs:
            draws += 1
        else:
            losses += 1
        if verbose:
            print(f"  game {g} (seat {int(swap)}): ${mine:>10,.0f}  vs  ${theirs:>10,.0f}"
                  f"   {'WIN ' if mine > theirs else ('TIE ' if mine == theirs else 'LOSS')}")
    return dict(
        games=games, wins=wins, draws=draws, losses=losses,
        winrate=round(wins / games, 3),
        my_mean=round(statistics.mean(my_banks)),
        my_median=round(statistics.median(my_banks)),
        my_min=round(min(my_banks)), my_max=round(max(my_banks)),
        opp_mean=round(statistics.mean(opp_banks)),
    )


res = duel("main.py", "starter", games=4)
print()
print(res)

### Debugging a broken agent

If an episode errors, the environment swallows it. Get the traceback like this:

In [ ]:
env = make("kaggriculture", configuration={"episodeSteps": 240}, debug=True)
env.run(["main.py", "pass"])
for i, s in enumerate(env.steps[-1]):
    print(i, s["status"])
# with debug=True, exceptions from your agent are printed above this line

### Instrumenting: where is my money coming from?

Add a tracer to your agent and print a per-day report. This is how you find out that you're spending
\$800/day on wheat you don't need, or throwing 40 eggs a day into the shed-overflow bin.

In [ ]:
import collections
from kaggle_environments import make


def trace_episode(agent_file="main.py", opponent="starter", steps=720):
    env = make("kaggriculture", configuration={"episodeSteps": steps})
    env.run([agent_file, opponent])
    hist = []
    for st in env.steps:
        o = st[0]["observation"]
        if "farms" not in o:
            continue
        hist.append(dict(step=o["step"], day=o["day"],
                         money=o["farms"][0]["money"],
                         opp=o["farms"][1]["money"],
                         **{f"px_{k}": v for k, v in o["market"]["prices"].items()}))
    return pd.DataFrame(hist)


tr = trace_episode(steps=720)
daily = tr.groupby("day").last()
ax = daily[["money", "opp"]].plot(figsize=(11, 4), lw=2, title="Bank balance by day")
ax.set_ylabel("$"); ax.grid(alpha=.3); plt.show()

px_cols = [c for c in daily.columns if c.startswith("px_")]
ax = daily[px_cols].plot(figsize=(11, 4), title="Market prices by day")
ax.set_ylabel("$"); ax.grid(alpha=.3); plt.show()

---
## 12. Submitting — and the pitfalls that break submissions

### The mechanics

Your submission is a **`main.py` at the root** defining a callable named `agent` (a module-level
`def agent(obs)` is the safe form). Multi-file agents go in a `.tar.gz` with `main.py` at the root.

```bash
pip install kaggle
# token: https://www.kaggle.com/settings/api  ->  ~/.kaggle/access_token
kaggle competitions submit kaggriculture -f main.py -m "v1: baseline"
kaggle competitions submissions kaggriculture
```

- **5 submissions/day**, **only your latest 2 stay active** and are used for the final leaderboard.
- On upload a **validation episode** runs your agent against itself. Any exception → `Error`.
- Files land in `/kaggle_simulations/agent/` — if you read a data file, resolve the path relative to
  `os.path.dirname(os.path.abspath(__file__))`, never the CWD.
- Limits: 100 MiB, 6.5 GiB RAM, 1.6 vCPU.

### The nine things that actually go wrong

1. 🔍 **`agent` is not the function that gets called — the LAST callable in the file is.**
   This is the single nastiest gotcha in the whole competition. From `kaggle_environments/agent.py`:
   ```python
   exec(code_object, env)
   return [v for v in env.values() if callable(v)][-1]     # <-- the LAST one
   ```
   Define one innocent helper below `def agent(...)` and the framework calls **the helper** instead,
   with the observation as its first argument. You get a `TypeError`, or worse, a silent wrong answer.
   **Keep `agent` as the final `def` in `main.py`**, and don't do `from math import sqrt` after it either.
2. **Returning the wrong number of hand ops.** `hands` must have exactly one entry per hired hand,
   in order. Recompute `len(me["hands"])` *every turn* — it changes, and hands hired this turn only
   appear next turn (market orders resolve after actions).
3. **Hands stuck on LOCKED shed tiles.** See §2.3 — the first hand you hire spawns on `(5,4)`, which is
   locked until you buy the NE quadrant. On a locked tile only movement works.
4. **Module-level state that isn't reset.** Your agent process may be reused across episodes. Key any
   cache on something episode-specific, or reset when `obs["step"] == 0`.
5. **Assuming `obs["step"]` exists.** Be defensive: `obs.get("step", obs["day"]*24 + obs["hour"])`.
6. **Walking off the board or into `"LOCKED"`.** Silent no-op; your unit stands still forever. Always
   bounds-check *and* locked-check before emitting a move. Note the quadrants can form an L-shape, so
   greedy Manhattan stepping can wedge a unit — use BFS.
7. **The atomic-PLANT rule.** Over-request a crop and *all* of that crop's plants are dropped, not just
   the excess. Reserve seeds across units within a turn.
8. **Ignoring the 100-item shed cap.** Overflow at the nightly dump is destroyed silently. If you
   harvest more than 100 units a day you must `DROP` and `SELL` *during* the day.
9. **>10 market orders.** Extras are dropped silently, and it is usually your `SELL` that gets cut —
   `HIRE` spam at hour 0 is a classic way to starve your own sell orders. Order the list by importance
   and truncate to 10 yourself.

### A pre-flight check to run before every submission

In [ ]:
import importlib.util, sys, time
from kaggle_environments import make


def preflight(path="main.py"):
    ok = True

    # 1. imports cleanly and exposes `agent`
    spec = importlib.util.spec_from_file_location("subm", path)
    mod = importlib.util.module_from_spec(spec)
    t0 = time.time()
    spec.loader.exec_module(mod)
    print(f"[ok] imports in {time.time()-t0:.2f}s")
    if not callable(getattr(mod, "agent", None)):
        print("[FAIL] no module-level callable named `agent`"); ok = False

    # 2. self-play validation episode (this is what Kaggle runs)
    t0 = time.time()
    env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
    env.run([path, path])
    dur = time.time() - t0
    st = [s["status"] for s in env.steps[-1]]
    print(f"[{'ok' if all(s == 'DONE' for s in st) else 'FAIL'}] self-play status {st}  ({dur:.1f}s)")
    if not all(s == "DONE" for s in st):
        ok = False

    # 3. per-turn latency
    print(f"[info] ~{dur/720*1000:.1f} ms per turn per agent")

    # 4. beats the baselines
    for opp in ("pass", "random", "starter"):
        env = make("kaggriculture", configuration={"episodeSteps": 720})
        env.run([path, opp])
        r = [s["reward"] or 0 for s in env.steps[-1]]
        verdict = "ok" if r[0] > r[1] else "FAIL"
        if r[0] <= r[1]:
            ok = False
        print(f"[{verdict}] vs {opp:8} ${r[0]:>12,.0f}  vs  ${r[1]:>12,.0f}")

    print("\nPREFLIGHT", "PASSED ✅" if ok else "FAILED ❌")
    return ok


preflight("main.py")

---
## 13. Where to go next

You now have the full rule set, the price function, the market-depth table, the labour curve, the
action budget, a working agent, and a test harness. That is everything you need to start iterating.

The gap between this baseline and a competitive agent is not one clever trick — it is a handful of
ordinary decisions, each made deliberately instead of by accident:

1. **When to buy land.** \$7,000 buys 4x the tiles and it compounds. Work out your own payback period.
2. **What to plant, and how much of it.** §4 gives you coins-per-tile-day and coins-per-action; §6
   tells you how much the market will actually absorb. You need both numbers, not either one.
3. **How much labour.** Hire at hour 0, and scale with the workload rather than the bank balance.
4. **How, and when, to sell.** Re-read §6: revenue is `Σ price(I0 + k)`, not `n x price(I0)`.
5. **Routing.** Instrument it before you optimise it — count how many of your unit-turns are `MOVE`.
   The number will probably surprise you.

### How to actually improve

The single most useful habit: **count things**. Wrap your agent, log the ops it emits per day, and
look at the census. Most of the big early wins are not strategic — they are a scheduler quietly
burning most of your labour, or the shed silently binning a third of your harvest.

And test properly: many seeds, both seats, and — once you have two candidate agents — play them
against **each other**, not just against `starter`. A baseline that never sells is not competing with
you for market depth, so it will happily tell you that strategies are good when they are not.

Happy farming. 🌾